# CBED Frame Simulation

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from cbed_simulation.frame_builder import FrameParameters
from cbed_simulation.crystal_orientation import OrientedPhase, ExperimentInformation

In [ ]:
phase = OrientedPhase.from_cif("./Si.cif", (1, 1, 0))

In [ ]:
experiment = ExperimentInformation(
    frame_shape=(512, 512),
    radius_px=10,
    transmitted_centre_px=complex(256, 256),
    pattern_scale_factor=120.,
)

In [ ]:
frame_params = FrameParameters(
    disk_blur_sigma=0.5,
    intensity_from_radius=False,
    textured=True,
    inelastic_scatter_sigma=0.,
    additive_noise_scale=0,
)

In [ ]:
sim_peaks = phase.peak_positions(experiment, stretch_abc=(1., 1., 1.), max_excitation_error=1e-5)
sim_peaks.modify_intensities(power=0.25)
sim_peaks.modify_000_intensity(multiply=2)
sim_peaks.weights.size

In [ ]:
frame_vacuum = phase.synthetic(experiment, sim_peaks, frame_params=frame_params)

In [ ]:
sim_peaks = phase.peak_positions(experiment, stretch_abc=(1., 1., 1.))
sim_peaks.modify_intensities(power=0.25)
sim_peaks.modify_000_intensity(multiply=2)

In [ ]:
frame_unstrained = phase.synthetic(experiment, sim_peaks, frame_params=frame_params)

In [ ]:
sim_peaks = phase.peak_positions(experiment, stretch_abc=(1., 1., 1.015))
sim_peaks.modify_intensities(power=0.25)
sim_peaks.modify_000_intensity(multiply=2)

In [ ]:
frame_strained = phase.synthetic(experiment, sim_peaks, frame_params=frame_params)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(frame_unstrained, cmap="gray")
ax2.imshow(frame_strained, cmap="gray")

## Validate strain values

In [ ]:
import libertem.api as lt
from libertem_blobfinder.common.correlation import get_peaks
from libertem_blobfinder.common.patterns import UserTemplate
from libertem_blobfinder.udf.correlation import FullFrameCorrelationUDF
from libertem_blobfinder.common.gridmatching import Matcher
from strain_mapping.strain_decomposition import compute_strain_large_def


def to_complex(yx) -> complex:
    return complex(yx[1], yx[0])


def match_template(ctx, frame, template, peaks):
    ds = ctx.load("memory", data=frame[np.newaxis, ...])
    udf = FullFrameCorrelationUDF(
        peaks=peaks.astype(int), match_pattern=template, zero_shift=None, upsample=20,
    )
    corr_result = ctx.run_udf(
        dataset=ds,
        udf=udf
    )
    positions = corr_result["refineds"].data.squeeze(axis=0)
    positions_int = corr_result["centers"].data.squeeze(axis=0)
    return positions, positions_int

In [ ]:
cx, cy = int(experiment.transmitted_centre_px.real), int(experiment.transmitted_centre_px.imag)
rad = int(experiment.radius_px + 4)
template = frame_vacuum[cy - rad: cy + rad + 1, cx - rad: cx + rad + 1]
template = np.clip(template, 0, 20)
template = UserTemplate(template)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(template.template, cmap='gray')

In [ ]:
ctx = lt.Context.make_with("inline")

In [ ]:
peaks_unstrained = get_peaks(frame_unstrained, template, 40)
peaks_strained = get_peaks(frame_strained, template, 40)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(frame_unstrained, cmap='gray')
ax1.plot(peaks_unstrained[:, 1], peaks_unstrained[:, 0], 'ro')
ax1.plot(peaks_strained[:, 1], peaks_strained[:, 0], 'go', alpha=0.3)
ax2.imshow(frame_strained, cmap='gray')
ax2.plot(peaks_strained[:, 1], peaks_strained[:, 0], 'go')
ax2.plot(peaks_unstrained[:, 1], peaks_unstrained[:, 0], 'ro', alpha=0.3)

In [ ]:
positions_unstrained, positions_unstrained_int = match_template(ctx, frame_unstrained, template, peaks_unstrained)
positions_strained, positions_strained_int = match_template(ctx, frame_strained, template, peaks_strained)

In [ ]:
zero = np.asarray((cy, cx))
vec_a = np.asarray((286, 286)) - zero
vec_b = np.asarray((248, 292)) - zero
match_unstrained = Matcher().fastmatch(positions_unstrained_int, zero, vec_a, vec_b, refineds=positions_unstrained)
match_strained = Matcher().fastmatch(positions_strained_int, zero, vec_a, vec_b, refineds=positions_strained)
match_strained.zero

In [ ]:
strain_res = compute_strain_large_def(
    to_complex(match_strained.a),
    to_complex(match_strained.b),
    to_complex(match_unstrained.a),
    to_complex(match_unstrained.b),
)
strain_res.to_vector(to_complex(match_strained.a))